# 1. Environment Setup & Data Loading
Welcome to the Model Training Pipeline! In this module, we will set up our environment by installing the required machine learning and deep learning libraries. We will then load the **balanced dataset** that was pre-processed, cleaned, and undersampled in our previous data pipeline notebook.

In [ ]:
# =====================================================
# MOUNT GOOGLE DRIVE
# =====================================================
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# =====================================================
# INSTALL REQUIRED LIBRARIES
# =====================================================
# We use -q (quiet) to keep the notebook output clean
!pip install transformers datasets accelerate evaluate rapidfuzz -q
!pip install scikit-learn matplotlib seaborn -q

print("All libraries successfully installed!")

In [ ]:
# =====================================================
# IMPORT LIBRARIES & LOAD DATA
# =====================================================
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

# Set the path to where we saved the balanced dataset
INPUT_FOLDER = Path("/content/drive/MyDrive/Colab Notebooks/TNL/Project/PREPROCESS DATASET")
INPUT_FILE = INPUT_FOLDER / "14_final_training_data.json"

print(f"Loading data from:\n{INPUT_FILE}\n")

# Load the JSON into a Pandas DataFrame
df = pd.read_json(INPUT_FILE)

print("=" * 60)
print(f"Dataset Loaded Successfully! Total Samples: {len(df):,}")
print("=" * 60)

# Verify the balancing worked correctly
print("\nLabel Distribution (Should be perfectly balanced):")
print(df["final_sentiment"].value_counts())

# Preview the data we will use for training
print("\nData Preview:")
display(df[["translated_sentence", "final_sentiment"]].head())

## 1.5 Unified Global Splitting & Saving
To ensure a fair comparison across all models, we perform a unified stratified split (70% Train, 15% Validation, 15% Test). We save these splits into a dedicated "SPLITS" folder so that every model (Logistic Regression, Naive Bayes, and RoBERTa) trains and evaluates on the exact same data.

In [ ]:
# =====================================================
# 1.5 UNIFIED GLOBAL SPLITTING & SAVING
# =====================================================
from pathlib import Path
from sklearn.model_selection import train_test_split

# Define the directory for split files
SPLITS_FOLDER = INPUT_FOLDER / "SPLITS"
SPLITS_FOLDER.mkdir(parents=True, exist_ok=True)

# Perform stratified split (Ensures 1:1:1 balance is kept in all sets)
# First: Split into 70% Train and 30% Temp (Val + Test)
df_train, df_temp = train_test_split(
    df,
    test_size=0.30,
    random_state=42,
    stratify=df["final_sentiment"]
)

# Second: Split Temp into 15% Val and 15% Test (50/50 split of the 30%)
df_val, df_test = train_test_split(
    df_temp,
    test_size=0.50,
    random_state=42,
    stratify=df_temp["final_sentiment"]
)

# Save the splits
df_train.to_json(SPLITS_FOLDER / "train.json", orient='records', indent=2)
df_val.to_json(SPLITS_FOLDER / "val.json", orient='records', indent=2)
df_test.to_json(SPLITS_FOLDER / "test.json", orient='records', indent=2)

print("--- Unified Data Splits Completed ---")
print(f"Training Set   : {len(df_train)} samples")
print(f"Validation Set : {len(df_val)} samples")
print(f"Testing Set    : {len(df_test)} samples")
print(f"\nFiles saved to folder: {SPLITS_FOLDER}")

## 2. TF-IDF Vectorization
In this module, we transform our cleaned text data into numerical vectors. We use the **TF-IDF (Term Frequency-Inverse Document Frequency)** algorithm, which helps the model prioritize unique, important words while de-emphasizing common, less informative words.

To ensure the integrity of our experiment, we:
1. **Fit** the vectorizer **only on the training set**.
2. **Transform** the validation and testing sets using the vocabulary learned from the training set. This is a critical step to prevent **data leakage**.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. Load the splits created in Module 1.5
SPLITS_FOLDER = INPUT_FOLDER / "SPLITS"
df_train = pd.read_json(SPLITS_FOLDER / "train.json")
df_val = pd.read_json(SPLITS_FOLDER / "val.json")
df_test = pd.read_json(SPLITS_FOLDER / "test.json")

# 2. Extract features and labels for Traditional ML
# Note: Traditional models work best with your processed 'lemma_sentence'
X_train = df_train["lemma_sentence"]
y_train = df_train["final_sentiment"]

X_val = df_val["lemma_sentence"]
y_val = df_val["final_sentiment"]

X_test = df_test["lemma_sentence"]
y_test = df_test["final_sentiment"]

# 3. Vectorize (TF-IDF)
# We use n-gram range (1,2) to capture word combinations like "not good"
vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=2)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_val_tfidf = vectorizer.transform(X_val)
X_test_tfidf = vectorizer.transform(X_test)

print("--- TF-IDF Vectorization Complete ---")
print(f"Training features shape   : {X_train_tfidf.shape}")
print(f"Validation features shape : {X_val_tfidf.shape}")
print(f"Testing features shape    : {X_test_tfidf.shape}")

## 3. Logistic Regression Training
Logistic Regression is a robust baseline model for sentiment classification. Here, we use `GridSearchCV` to perform 5-fold cross-validation, testing different regularization strengths (`C`) to find the optimal model configuration that maximizes the Macro F1 score.

In [ ]:
# =====================================================
# MODULE 3: LOGISTIC REGRESSION (WITH GRID SEARCH)
# =====================================================
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# 1. Initialize Model
model_lr = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)

# 2. Define Hyperparameter Grid
param_grid_lr = {"C": [0.01, 0.1, 1, 10, 100]}

# 3. Setup Grid Search
grid_lr = GridSearchCV(
    estimator=model_lr,
    param_grid=param_grid_lr,
    cv=5,
    scoring="f1_macro",
    n_jobs=-1
)

# 4. Train
print("Training Logistic Regression...")
grid_lr.fit(X_train_tfidf, y_train)

# 5. Predict on Test Set
best_lr = grid_lr.best_estimator_
y_pred_lr = best_lr.predict(X_test_tfidf)

# 6. Output Results
print("\n" + "="*40)
print("LOGISTIC REGRESSION RESULTS")
print("="*40)
print(f"Best Parameters: {grid_lr.best_params_}")
print(f"Accuracy       : {accuracy_score(y_test, y_pred_lr):.4f}\n")
print("Classification Report:")
print(classification_report(y_test, y_pred_lr))

# 7. Plot Confusion Matrix
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_lr, cmap="Blues")
plt.title("Logistic Regression - Confusion Matrix")
plt.show()

## 4. Multinomial Naive Bayes Training
Naive Bayes is highly efficient for text data. Given its probabilistic nature, it is often a great complement to Logistic Regression. We will tune the smoothing parameter (`alpha`) to ensure the model generalizes well to unseen data.

In [ ]:
# =====================================================
# MODULE 4: MULTINOMIAL NAIVE BAYES (WITH GRID SEARCH)
# =====================================================
from sklearn.naive_bayes import MultinomialNB

# 1. Initialize Model
model_nb = MultinomialNB()

# 2. Define Hyperparameter Grid
param_grid_nb = {"alpha": [0.01, 0.1, 0.5, 1.0, 2.0]}

# 3. Setup Grid Search
grid_nb = GridSearchCV(
    estimator=model_nb,
    param_grid=param_grid_nb,
    cv=5,
    scoring="f1_macro",
    n_jobs=-1
)

# 4. Train
print("Training Multinomial Naive Bayes...")
grid_nb.fit(X_train_tfidf, y_train)

# 5. Predict on Test Set
best_nb = grid_nb.best_estimator_
y_pred_nb = best_nb.predict(X_test_tfidf)

# 6. Output Results
print("\n" + "="*40)
print("NAIVE BAYES RESULTS")
print("="*40)
print(f"Best Parameters: {grid_nb.best_params_}")
print(f"Accuracy       : {accuracy_score(y_test, y_pred_nb):.4f}\n")
print("Classification Report:")
print(classification_report(y_test, y_pred_nb, digits=4))

# 7. Plot Confusion Matrix
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_nb, cmap="Oranges")
plt.title("Naive Bayes - Confusion Matrix")
plt.show()

## 5. Transformer Data Preparation
Transformers require data in a specific format: Hugging Face `Dataset` objects. We will take the `train`, `val`, and `test` splits we created in Module 1.5, map the sentiment strings to integer IDs, and use the RoBERTa tokenizer to convert the raw text into token IDs and attention masks.

In [ ]:
# =====================================================
# MODULE 5: PREPARE DATASETS FOR HUGGINGFACE
# =====================================================
from datasets import Dataset
from transformers import AutoTokenizer

# 1. Load splits
df_train = pd.read_json(SPLITS_FOLDER / "train.json")
df_val = pd.read_json(SPLITS_FOLDER / "val.json")
df_test = pd.read_json(SPLITS_FOLDER / "test.json")

# 2. Map sentiment strings to integer IDs
label2id = {"negative": 0, "neutral": 1, "positive": 2}
id2label = {0: "negative", 1: "neutral", 2: "positive"}

# 3. Create Hugging Face Dataset objects
train_dataset = Dataset.from_dict({
    "text": df_train["translated_sentence"].tolist(),
    "label": df_train["final_sentiment"].map(label2id).tolist()
})
val_dataset = Dataset.from_dict({
    "text": df_val["translated_sentence"].tolist(),
    "label": df_val["final_sentiment"].map(label2id).tolist()
})
test_dataset = Dataset.from_dict({
    "text": df_test["translated_sentence"].tolist(),
    "label": df_test["final_sentiment"].map(label2id).tolist()
})

# 4. Tokenization
MODEL_NAME = "cardiffnlp/twitter-roberta-base-sentiment-latest"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_batch(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=128)

train_dataset = train_dataset.map(tokenize_batch, batched=True)
val_dataset = val_dataset.map(tokenize_batch, batched=True)
test_dataset = test_dataset.map(tokenize_batch, batched=True)

train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
val_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

print("Transformer Datasets (Train/Val/Test) are ready for training.")

## 6. Fine-Tune RoBERTa
We will now use the Hugging Face `Trainer` API to fine-tune RoBERTa. This process updates the model's weights to better understand your specific product review vocabulary. We will use Macro F1 as our primary metric to ensure we are correctly balancing performance across all three sentiment classes.

In [ ]:
# !pip install --upgrade torchvision

In [ ]:
# !pip uninstall -y torch torchvision torchaudio transformers
# !pip install torch torchvision torchaudio transformers accelerate datasets evaluate

In [ ]:
# =====================================================
# MODULE 6: TRAINING CONFIGURATION & FINE-TUNING
# =====================================================
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import evaluate
import numpy as np
from transformers import EarlyStoppingCallback

# Load model
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=3, id2label=id2label, label2id=label2id
)

# Define metrics
f1 = evaluate.load("f1")
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    macro_f1 = f1.compute(predictions=predictions, references=labels, average="macro")
    return {"macro_f1": macro_f1["f1"]}

# Training arguments
training_args = TrainingArguments(
    output_dir="./roberta_output",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    num_train_epochs=30,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print("Starting RoBERTa Fine-Tuning...")
trainer.train()

## 7. Evaluation and Export
Now that RoBERTa is fine-tuned, we evaluate its performance on the `test_dataset`. We also save the model locally and zip it so you can download your fine-tuned AI model directly to your computer.

In [ ]:
# =====================================================
# EVALUATE ON TEST SET & SAVE MODEL
# =====================================================
from sklearn.metrics import classification_report, confusion_matrix

print("Evaluating RoBERTa on the test set...")
pred_results = trainer.predict(test_dataset)
y_pred_tf = np.argmax(pred_results.predictions, axis=1)

# Display classification report
y_test_tf = test_dataset["label"]

print("\n--- RoBERTa Fine-Tuned Results ---")
print(classification_report(y_test_tf, y_pred_tf, target_names=["negative", "neutral", "positive"]))

# Plot Confusion Matrix
fig, ax = plt.subplots(figsize=(6, 6))
ConfusionMatrixDisplay.from_predictions(
    y_test_tf,
    y_pred_tf,
    cmap="Greens",
    display_labels=["negative", "neutral", "positive"],
    ax=ax
)

ax.grid(False)

plt.title("RoBERTa - Confusion Matrix")
plt.show()

# Save model
save_path = "./my_best_roberta_model"
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

# Zip and download
import shutil
from google.colab import files
shutil.make_archive("my_model", 'zip', save_path)
print("Model zipped. Initiating download...")
files.download("my_model.zip")

In [ ]:
# Plot Confusion Matrix
fig, ax = plt.subplots(figsize=(6, 6))
ConfusionMatrixDisplay.from_predictions(
    y_test_tf,
    y_pred_tf,
    cmap="Greens",
    display_labels=["negative", "neutral", "positive"],
    ax=ax
)

ax.grid(False)

plt.title("RoBERTa - Confusion Matrix")
plt.show()

## 8. Final Model Comparison
We aggregate the metrics from Logistic Regression, Naive Bayes, and RoBERTa to visualize which model performed best. This grouped bar chart provides a clear overview of your project's effectiveness.

In [ ]:
# =====================================================
# MODEL COMPARISON GRAPH
# =====================================================
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import f1_score

# 1. Gather Metrics
results = {
    "Model": ["Logistic Regression", "Naive Bayes", "RoBERTa"],
    "Accuracy": [
        accuracy_score(y_test, y_pred_lr),
        accuracy_score(y_test, y_pred_nb),
        accuracy_score(y_test_tf, y_pred_tf)
    ],
    "Macro F1": [
        f1_score(y_test, y_pred_lr, average="macro"),
        f1_score(y_test, y_pred_nb, average="macro"),
        f1_score(y_test_tf, y_pred_tf, average="macro")
    ]
}

df_results = pd.DataFrame(results)

# 2. Visualize
df_melted = df_results.melt(id_vars="Model", var_name="Metric", value_name="Score")
sns.set_theme(style="whitegrid")
plt.figure(figsize=(10, 6))

ax = sns.barplot(data=df_melted, x="Model", y="Score", hue="Metric", palette="viridis")

plt.title("Sentiment Analysis: Model Performance Comparison", fontsize=16, fontweight='bold')
plt.ylim(0, 1.1)

# Annotate values
for p in ax.patches:
    ax.annotate(format(p.get_height(), '.3f'),
                   (p.get_x() + p.get_width() / 2., p.get_height()),
                   ha = 'center', va = 'center', xytext = (0, 10), textcoords = 'offset points')

plt.tight_layout()
plt.savefig("model_comparison_chart.png", dpi=300)
plt.show()